In [ ]:
import pandas as pd
import os


In [ ]:
### command line
# cat {anc}*varlists > {anc}_all.varlist
# awk '{print $1}' {anc}_all.varlist | sort | uniq > tmp && mv tmp {anc}_all.varlist # keeping only unique vars

# get var info file
# plink2 --pfile ../merged_{anc} --extract {anc}_all.varlist --export A --out {anc}_var_matrix 

Adjustments are done separately for each mph output file. Adjustments correct per-GRM heritability and fold-enrichment estimates for variant overlaps between GRMs. Adjustments take into account the variant overlap, and the weights associated with that variant's contribution to GRM calculation (relative to its MAF, since GCTA-alpha uses the MAF-dependent alpha parameter to build GRMs).
<br/><br/>
As specific `sw`, `si`, and `cp` matrices correspond to specific GRM combinations used, for example heritability and fold-enrichment adjustments for height and BMI using the 34 GRMs  corresponding to X chr + autosomes including ncRNA subcategories use the same `sw`/`si`/`cp` matrices.

In [ ]:
all_vars = pd.read_csv(f"{anc}_all.varlist", names=['SNP'])
all_vars

## Making `si` (variant incidence matrix) per functional annotation bin

In [ ]:
# coding

# load in varlists
all_vars = pd.read_csv(f"{anc}_all.varlist", names=["SNP"])
rare_low = pd.read_csv(f"{anc}_newvep01_coding_lowbin_1e-4_0.01.varlist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep02_coding_highbin_1e-4_0.01.varlist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep03_coding_lowbin_0.01_0.5.varlist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep04_coding_highbin_0.01_0.5.varlist", header=0, names=["SNP"])

dfs = {"all_vars": all_vars,
       f"{anc}_newvep01_coding_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep02_coding_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep03_coding_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep04_coding_highbin_0.01_0.5": comm_high}

# stack all varlists by variant
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build variant incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

matrix.to_csv(f"{anc}_coding_vars.si", index=False)

In [ ]:
# ncRNA
# load in varlists
all_vars = pd.read_csv(f"{anc}_all.varlist", names=["SNP"])
rare_low = pd.read_csv(f"{anc}_newvep05_rna_lowbin_1e-4_0.01.varlist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep06_rna_highbin_1e-4_0.01.varlist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep07_rna_lowbin_0.01_0.5.varlist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep08_rna_highbin_0.01_0.5.varlist", header=0, names=["SNP"])

dfs = {"all_vars": all_vars,
       f"{anc}_newvep05_rna_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep06_rna_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep07_rna_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep08_rna_highbin_0.01_0.5": comm_high}

# stack all varlists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build var incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv(f"{anc}_rna_vars.si", index=False)

In [ ]:
# prox UTR
# load in snplists
rare_low = pd.read_csv(f"{anc}_newvep09_proximal_utr_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep10_proximal_utr_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep11_proximal_utr_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep12_proximal_utr_highbin_0.01_0.5.snplist", header=0, names=["SNP"])

dfs = {"all_vars": all_snps,
       f"{anc}_newvep09_proximal_utr_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep10_proximal_utr_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep11_proximal_utr_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep12_proximal_utr_highbin_0.01_0.5": comm_high}

# stack all snplists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build snp incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv("prox_utr_snps.si", index=False)

In [ ]:
# prox not UTR
# load in snplists
rare_low = pd.read_csv(f"{anc}_newvep13_proximal_notutr_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep14_proximal_notutr_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep15_proximal_notutr_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep16_proximal_notutr_highbin_0.01_0.5.snplist", header=0, names=["SNP"])

dfs = {"all_vars": all_snps,
       f"{anc}_newvep13_proximal_notutr_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep14_proximal_notutr_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep15_proximal_notutr_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep16_proximal_notutr_highbin_0.01_0.5": comm_high}

# stack all snplists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build snp incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv("prox_notutr_snps.si", index=False)

In [ ]:
# intronic
# load in snplists
rare_low = pd.read_csv(f"{anc}_newvep17_intronic_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep18_intronic_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep19_intronic_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep20_intronic_highbin_0.01_0.5.snplist", header=0, names=["SNP"])

dfs = {"all_vars": all_snps,
       f"{anc}_newvep17_intronic_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep18_intronic_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep19_intronic_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep20_intronic_highbin_0.01_0.5": comm_high}

# stack all snplists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build snp incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv("intronic_snps.si", index=False) 

In [ ]:
# regulatory
# load in snplists
all_snps = pd.read_csv(f"{anc}_all.snplist", names=["SNP"])
rare_low = pd.read_csv(f"{anc}_newvep21_regulatory_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep22_regulatory_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep23_regulatory_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep24_regulatory_highbin_0.01_0.5.snplist", header=0, names=["SNP"])

dfs = {"all_vars": all_snps,
       f"{anc}_newvep21_regulatory_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep22_regulatory_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep23_regulatory_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep24_regulatory_highbin_0.01_0.5": comm_high}

# stack all snplists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build snp incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv("regulatory_snps.si", index=False) 

In [ ]:
# other
# load in snplists
rare_low = pd.read_csv(f"{anc}_newvep25_other_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
rare_high = pd.read_csv(f"{anc}_newvep26_other_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
comm_low = pd.read_csv(f"{anc}_newvep27_other_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
comm_high = pd.read_csv(f"{anc}_newvep28_other_highbin_0.01_0.5.snplist", header=0, names=["SNP"])

dfs = {"all_vars": all_snps,
       f"{anc}_newvep25_other_lowbin_1e-4_0.01": rare_low, 
       f"{anc}_newvep26_other_highbin_1e-4_0.01": rare_high, 
       f"{anc}_newvep27_other_lowbin_0.01_0.5": comm_low, 
       f"{anc}_newvep28_other_highbin_0.01_0.5": comm_high}

# stack all snplists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build snp incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv("other_snps.si", index=False) 

In [ ]:
# sep ncRNA
bin_29 = pd.read_csv(f"{anc}_newvep29_overlap_lncRNA_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_30 = pd.read_csv(f"{anc}_newvep30_overlap_lncRNA_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_31 = pd.read_csv(f"{anc}_newvep31_overlap_lncRNA_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_32 = pd.read_csv(f"{anc}_newvep32_overlap_lncRNA_highbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_33 = pd.read_csv(f"{anc}_newvep33_overlap_miRNA_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_34 = pd.read_csv(f"{anc}_newvep34_overlap_miRNA_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_35 = pd.read_csv(f"{anc}_newvep35_overlap_miRNA_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_36 = pd.read_csv(f"{anc}_newvep36_overlap_miRNA_highbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_37 = pd.read_csv(f"{anc}_newvep37_overlap_rRNA_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_38 = pd.read_csv(f"{anc}_newvep38_overlap_rRNA_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_39 = pd.read_csv(f"{anc}_newvep39_overlap_rRNA_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_40 = pd.read_csv(f"{anc}_newvep40_overlap_rRNA_highbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_41 = pd.read_csv(f"{anc}_newvep41_overlap_snoRNA_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_42 = pd.read_csv(f"{anc}_newvep42_overlap_snoRNA_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_43 = pd.read_csv(f"{anc}_newvep43_overlap_snoRNA_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_44 = pd.read_csv(f"{anc}_newvep44_overlap_snoRNA_highbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_45 = pd.read_csv(f"{anc}_newvep45_overlap_snRNA_lowbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_46 = pd.read_csv(f"{anc}_newvep46_overlap_snRNA_highbin_1e-4_0.01.snplist", header=0, names=["SNP"])
bin_47 = pd.read_csv(f"{anc}_newvep47_overlap_snRNA_lowbin_0.01_0.5.snplist", header=0, names=["SNP"])
bin_48 = pd.read_csv(f"{anc}_newvep48_overlap_snRNA_highbin_0.01_0.5.snplist", header=0, names=["SNP"])

dfs = {"all_vars": all_snps,
       f"{anc}_newvep29_overlap_lncRNA_lowbin_1e-4_0.0": bin_29, 
       f"{anc}_newvep30_overlap_lncRNA_highbin_1e-4_0.01": bin_30, 
       f"{anc}_newvep31_overlap_lncRNA_lowbin_0.01_0.5": bin_31, 
       f"{anc}_newvep32_overlap_lncRNA_highbin_0.01_0.5": bin_32,
       f"{anc}_newvep33_overlap_miRNA_lowbin_1e-4_0.01": bin_33, 
       f"{anc}_newvep34_overlap_miRNA_highbin_1e-4_0.01": bin_34, 
       f"{anc}_newvep35_overlap_miRNA_lowbin_0.01_0.5": bin_35, 
       f"{anc}_newvep36_overlap_miRNA_highbin_0.01_0.5": bin_36,
       f"{anc}_newvep37_overlap_rRNA_lowbin_1e-4_0.01": bin_37, 
       f"{anc}_newvep38_overlap_rRNA_highbin_1e-4_0.01": bin_38, 
       f"{anc}_newvep39_overlap_rRNA_lowbin_0.01_0.5": bin_39, 
       f"{anc}_newvep40_overlap_rRNA_highbin_0.01_0.5": bin_40,
       f"{anc}_newvep41_overlap_snoRNA_lowbin_1e-4_0.01": bin_41, 
       f"{anc}_newvep42_overlap_snoRNA_highbin_1e-4_0.01": bin_42, 
       f"{anc}_newvep43_overlap_snoRNA_lowbin_0.01_0.5": bin_43, 
       f"{anc}_newvep44_overlap_snoRNA_highbin_0.01_0.5": bin_44,
       f"{anc}_newvep45_overlap_snRNA_lowbin_1e-4_0.01": bin_45, 
       f"{anc}_newvep46_overlap_snRNA_highbin_1e-4_0.01": bin_46, 
       f"{anc}_newvep47_overlap_snRNA_lowbin_0.01_0.5": bin_47, 
       f"{anc}_newvep48_overlap_snRNA_highbin_0.01_0.5": bin_48}

# stack all snplists by SNP
stacked = pd.concat(
    [df.assign(source=name) for name, df in dfs.items()],
    ignore_index=True
)

# build snp incidence matrix
matrix = pd.crosstab(stacked['SNP'], stacked['source'])
matrix = (matrix > 0).astype(int).reset_index()

# write matrix to file
matrix.to_csv(f"{anc}_sep_rna_snps.si", index=False) 

## Making `sw` (weighted variant incidence matrix) per functional annotation bin

In [ ]:
ldscores = pd.read_csv(f"../ldscores/{anc}_score.ld", sep=' ', header=None)
ldscores.columns = ['SNP', 'chr', 'bp', 'MAF', 'mean_rsq', 'snp_num', 'max_rsq', 'ldscore']
ldscores.drop(columns=['chr', 'bp', 'mean_rsq', 'snp_num', 'max_rsq', 'ldscore'])

In [ ]:

ldscore_subset = pd.merge(matrix['SNP'], ldscores, on='SNP', how='left')
ldscore_subset['wt'] = 2*ldscore_subset['MAF']*(1-ldscore_subset['MAF'])**(0.75)
ldscore_subset

In [ ]:
# per annotation
matrix = pd.read_csv(f"{anc}_coding_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_coding_vars.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_rna_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_rna_vars.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_prox_utr_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_prox_utr_vars.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_prox_notutr_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_prox_notutr_vars.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_intronic_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_intronic_vars.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_regulatory_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}regulatory_vars.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_other_snps.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_other_snps.sw", index=False, sep='\t')

In [ ]:
matrix = pd.read_csv(f"{anc}_sep_rna_vars.si")
matrix.iloc[:,1] *= ldscore_subset['wt']
matrix.iloc[:,2] *= ldscore_subset['wt']
matrix.iloc[:,3] *= ldscore_subset['wt']
matrix.iloc[:,4] *= ldscore_subset['wt']
matrix.iloc[:,5] *= ldscore_subset['wt']
matrix.iloc[:,6] *= ldscore_subset['wt']
matrix.iloc[:,7] *= ldscore_subset['wt']
matrix.iloc[:,8] *= ldscore_subset['wt']
matrix.iloc[:,9] *= ldscore_subset['wt']
matrix.iloc[:,10] *= ldscore_subset['wt']
matrix.iloc[:,11] *= ldscore_subset['wt']
matrix.iloc[:,12] *= ldscore_subset['wt']
matrix.iloc[:,13] *= ldscore_subset['wt']
matrix.iloc[:,14] *= ldscore_subset['wt']
matrix.iloc[:,15] *= ldscore_subset['wt']
matrix.iloc[:,16] *= ldscore_subset['wt']
matrix.iloc[:,17] *= ldscore_subset['wt']
matrix.iloc[:,18] *= ldscore_subset['wt']
matrix.iloc[:,19] *= ldscore_subset['wt']
matrix.iloc[:,20] *= ldscore_subset['wt']
matrix.to_csv(f"{anc}_sep_rna_vars.sw", index=False, sep='\t')

In [ ]:
!cat {anc}*vars.sw > {anc}_all_variants.sw

## Adjust enrichments for overlap

In [ ]:
# Read the variant info file to get the variant weighting matrix <sw>
sw = fread("../cat_annots/merged_afr_sepRNA_afternorm.sw")
head(sw)

In [ ]:
#bin_directory <- read.csv("annots/SNP_bin_directory.csv")
#bin_directory

In [ ]:
#main_norms <- bin_directory$AFR.alpha.normalisation..same.for.all.GRM.parts.[1:28]
#main_norms
#rna_norms <- bin_directory$AFR.alpha.normalisation..same.for.all.GRM.parts.[30:49]
#rna_norms